<img width="200" style="float:left" src="https://upload.wikimedia.org/wikipedia/commons/f/f3/Apache_Spark_logo.svg" />

# Sections
* [Description - Madrid's Air Quality Analysis (2018-2019)](#0)
* [1. Setup](#1)
  * [1.1 Start Hadoop](#1.1)
  * [1.2 Create SparkSession](#1.2)
* [2. Lab](#2)
  * [2.1 Spark data abstraction (DataFrame) for the CSV files](#2.1)
  * [2.2 Create a DataFrame containing reference data manually](#2.2)
  * [2.3 DataFrames combination (join) to enrich data coming from different sources](#2.3)
* [3. Going further (new)](#3)
  * [3.1 From wide to long](#3.1)
  * [3.2 NO₂ per station vs the EU limit](#3.2)
  * [3.3 Madrid Central: Pza. del Carmen](#3.3)
  * [3.4 The hourly limit](#3.4)
  * [3.5 Hours, weekdays and seasons](#3.5)
  * [3.6 Save the result as Parquet](#3.6)
* [4. TearDown](#4)
  * [4.1 Stop Spark and Hadoop](#4.1)

<a id='0'></a>
# Lab 16.3 — Processing Madrid's Air Quality (2018-2019)

## Description

Throughout the following Jupyter Notebook you're going to analyze the **air quality in Madrid** from **January 2018** to **November 2019**, by using the **Spark skills** you've learned during the **Spark Course**.

Data set used is coming from the [Madrid Town's Open Data Portal](https://datos.madrid.es/), specifically, from the dataset *Calidad del aire. Datos horarios desde 2001* (licence CC BY 4.0). **Documentation on the data set is available** in the *Intérprete de ficheros de calidad del aire* published with the dataset, although *it's in Spanish*. The document will be used as a reference and *translated as needed*.

**Prerequisite:** lab 16.2. The course files must be in HDFS: 12 files in `/datalake/raw/madrid/airqualitydata/2018` and 11 in `/datalake/raw/madrid/airqualitydata/2019` (there is no December 2019 in the course zip).

*Updated version (November 2026): same structure and code as the original, with the reference data corrected against the current interpreter and a new section 3 that goes further with the same data.*

<a id='1'></a>
# 1. Setup

Since we are going to process data stored from HDFS let's start the service

<a id='1.1'></a>
## 1.1 Start Hadoop

In the OSBDET WebUI (`localhost:2026`), switch **Hadoop** on and wait about 30 seconds: the NameNode starts in *safe mode* (read-only).

Pandas setup to display contents in a better way:

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

<a id='1.2'></a>
## 1.2 Create SparkSession
It's a good practice to give the Spark job a name and, before doing the actual data analysis, check the Spark version you're running to be sure that all the functions you'll use will be available and will behave as you expect.

While the session is alive, the Spark UI is at `localhost:24040`.

In [ ]:
from pyspark.sql.session import SparkSession

spark = SparkSession.builder\
                    .appName("Madrid's Air Quality Analysis")\
                    .getOrCreate()

print(f"This cluster relies on Spark '{spark.version}'")

<a id='2'></a>
# 2. Lab

<a id='2.1'></a>
## 2.1 Spark data abstraction (DataFrame) for the CSV files
Create a **custom schema** manually to translate CSV files headers (from Spanish to English):
  - PROVINCE, TOWN, STATION, MAGNITUDE => `IntegerType`
  - SAMPLING_LOCATION => `StringType`
  - YEAR, MONTH, DAY => `IntegerType`
  - H01 … H24 => `DoubleType` (hourly values)
  - V01 … V24 => `StringType` (validation flags: `V` valid, `N` not valid)

The names come from the schema **by position**, so Spark logs a harmless warning: *CSV header does not conform to the schema. Expected: PROVINCE but found: PROVINCIA*.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

aqSchema = StructType(\
    [StructField("PROVINCE",IntegerType(),True),\
     StructField("TOWN",IntegerType(),True),\
     StructField("STATION",IntegerType(),True),\
     StructField("MAGNITUDE",IntegerType(),True),\
     StructField("SAMPLING_LOCATION",StringType(),True),\
     StructField("YEAR",IntegerType(),True),\
     StructField("MONTH",IntegerType(),True),\
     StructField("DAY",IntegerType(),True)] +\
    [StructField(f"{t}{h:02d}", DoubleType() if t == "H" else StringType(), True)
     for h in range(1, 25) for t in "HV"])     # H01, V01, H02, V02 ... H24, V24

len(aqSchema.fields)

Let's create two DataFrames containing airquality data for 2018 and 2019, and let's consolidate them in one single DataFrame by unifying them:

In [ ]:
aqDF18 = spark.read.schema(aqSchema)\
                   .option("header", "true")\
                   .option("sep", ";")\
                   .csv("hdfs://localhost:9000/datalake/raw/madrid/airqualitydata/2018/*.csv")
aqDF19 = spark.read.schema(aqSchema)\
                   .option("header", "true")\
                   .option("sep", ";")\
                   .csv("hdfs://localhost:9000/datalake/raw/madrid/airqualitydata/2019/*.csv")
aqDF = aqDF18.union(aqDF19)     # by position; unionByName() matches by column name

Let's retrieve ***the first row to have a good sense of the data*** we're going to work with; let's calculate ***how many rows there are*** in the DataFrame as well:

In [ ]:
aqDF.first()

In [ ]:
print(f"This DataFrame has {aqDF.count()}")      # 105,050 = 54,831 (2018) + 50,219 (2019)

<a id='2.2'></a>
## 2.2 Create a DataFrame containing reference data manually
Let's create a DataFrame manually (we'll call it `magnitudesDF`) with the contents in the table **"Magnitudes, units and measurement techniques"**, available in **Appendix II** of the interpreter.

*Corrected against the current interpreter: PM2.5 is "Particles < 2.5 µm" (not 250 µm), "Metaxylene" spelling, hydrocarbons 42–44 measured by gas chromatography (technique 59), and the new code 431 (Metaparaxylene).*

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

magnitudes = [ Row(1, "Sulphur dioxide", "SO2", "ug/m^3", 38, "Ultraviolet fluorescence"),
               Row(6, "Carbon monoxide", "CO", "mg/m^3", 48, "Infrared absorption"),
               Row(7, "Nitrogen monoxide", "NO", "ug/m^3", 8, "Chemiluminescence"),
               Row(8, "Nitrogen dioxide", "NO2", "ug/m^3", 8, "Id."),
               Row(9, "Particles < 2.5um", "PM2.5", "ug/m^3", 47, "Microbalance / spectrometry"),
               Row(10, "Particles < 10um", "PM10", "ug/m^3", 47, "Id."),
               Row(12, "Nitrogen oxides", "NOx", "ug/m^3", 8, "Chemiluminescence"),
               Row(14, "Ozone", "O3", "ug/m^3", 6, "Ultraviolet absorption"),
               Row(20, "Toluene", "TOL", "ug/m^3", 59, "Gas chromatography"),
               Row(30, "Benzene", "BEN", "ug/m^3", 59, "Id."),
               Row(35, "Ethylbenzene", "EBE", "ug/m^3", 59, "Id."),
               Row(37, "Metaxylene", "MXY", "ug/m^3", 59, "Id."),
               Row(38, "Paraxylene", "PXY", "ug/m^3", 59, "Id."),
               Row(39, "Orthoxylene", "OXY", "ug/m^3", 59, "Id."),
               Row(42, "Total hydrocarbons (hexane)", "TCH", "mg/m^3", 59, "Id."),
               Row(43, "Methane", "CH4", "mg/m^3", 59, "Id."),
               Row(44, "Non-methane hydrocarbons (hexane)", "NMHC", "mg/m^3", 59, "Id."),
               Row(431, "Metaparaxylene", "MPX", "mg/m^3", 59, "Id.") ]
magnitudesSchema = StructType([ StructField("magnitudeId", IntegerType(), False),
                                StructField("magnitudeName", StringType(), True),
                                StructField("formula", StringType(), True),
                                StructField("unit", StringType(), True),
                                StructField("techniqueId", IntegerType(), True),
                                StructField("techniqueName", StringType(), True) ])
magnitudesDF = spark.createDataFrame(magnitudes, magnitudesSchema)
magnitudesDF.show(truncate=False)

<a id='2.3'></a>
## 2.3 DataFrames combination (join) to enrich data coming from different sources
We're going to create a Dataframe called `enrichedAqDF` combining `aqDF` and `magnitudesDF`, which displays all the columns in `aqDF` but replacing column `MAGNITUDE` by column `magnitudeName` available in `magnitudesDF`, and removing all pair columns from `H01`, `V01` to `H24`, `V24`.

**New:** `F.broadcast()` tells Spark that `magnitudesDF` is small. Without it, Spark broadcasts the *big* side (all of `aqDF`, 16 MiB), because a DataFrame built from Python `Row` objects has no size statistics. Compare query "toPandas" with and without the hint in the SQL / DataFrame tab of the Spark UI. With the hint, the 10 rows shown can differ from the original screenshot (we get station 4, December 2018): without `orderBy`, Spark doesn't promise which 10 rows `limit(10)` returns.

In [ ]:
from pyspark.sql import functions as F

enrichedAqDF = aqDF.join(F.broadcast(magnitudesDF.withColumnRenamed("magnitudeId","MAGNITUDE")),"MAGNITUDE")\
    .select("PROVINCE","TOWN","STATION","magnitudeName","SAMPLING_LOCATION","YEAR","MONTH","DAY")\
    .withColumnRenamed("magnitudeName", "MAGNITUDENAME")

In [ ]:
enrichedAqDF.printSchema()

In [ ]:
enrichedAqDF.limit(10).toPandas()

In [ ]:
print(f"Rows after the join: {enrichedAqDF.count()}")   # always count after a join: 105,050

<a id='3'></a>
# 3. Going further (new)
The same data answers more interesting questions once it's in a better shape.

<a id='3.1'></a>
## 3.1 From wide to long
One row per **hourly measurement** instead of one row per day with 24 value columns. `stack(24, 1, H01, V01, 2, H02, V02, …)` turns each row into 24 rows.

In [ ]:
hours = [f"{h:02d}" for h in range(1, 25)]
pairs = ", ".join(f"{int(h)}, H{h}, V{h}" for h in hours)
stackExpr = f"stack(24, {pairs}) as (HOUR, VALUE, VALID)"

longDF = aqDF.selectExpr("STATION", "MAGNITUDE", "YEAR", "MONTH", "DAY", stackExpr)
longDF.show(3)
longDF.groupBy("VALID").count().show()      # V 2,496,666 · N 24,534 (0.97 %)

In [ ]:
# From now on, validated values only. An 'N' value can be 0.0: e.g. the hour that doesn't
# exist on the spring DST change (25 March 2018).
validDF = longDF.where("VALID = 'V'").drop("VALID").cache()

<a id='3.2'></a>
## 3.2 NO₂ per station vs the EU limit
EU annual limit: **40 µg/m³** (Directive 2008/50/EC), **20 µg/m³ from 2030** (Directive (EU) 2024/2881); WHO 2021 guideline: 10. The course files stop in November 2019, so we compare **January–November** of both years.

In [ ]:
stations = [(4, "Pza. de España"), (8, "Escuelas Aguirre"), (11, "Av. Ramón y Cajal"),
            (16, "Arturo Soria"), (17, "Villaverde Alto"), (18, "C/ Farolillo"),
            (24, "Casa de Campo"), (27, "Barajas"), (35, "Pza. del Carmen"),
            (36, "Moratalaz"), (38, "Cuatro Caminos"), (39, "Barrio del Pilar"),
            (40, "Vallecas"), (47, "Méndez Álvaro"), (48, "Pº. Castellana"),
            (49, "Retiro"), (50, "Pza. Castilla"), (54, "Ensanche Vallecas"),
            (55, "Urb. Embajada (Barajas)"), (56, "Plaza Elíptica"), (57, "Sanchinarro"),
            (58, "El Pardo"), (59, "Parque Juan Carlos I"), (60, "Tres Olivos")]
stationsDF = spark.createDataFrame(stations, "STATION INT, STATION_NAME STRING")

no2DF = validDF.where("MAGNITUDE = 8")

annualDF = (no2DF.where("MONTH <= 11")              # like with like
    .groupBy("STATION").pivot("YEAR", [2018, 2019])
    .agg(F.round(F.avg("VALUE"), 1))
    .join(F.broadcast(stationsDF), "STATION")
    .withColumn("CHANGE", F.round(F.col("2019") - F.col("2018"), 1))
    .orderBy(F.desc("2019")))
annualDF.toPandas()

In [ ]:
annualDF.select(F.sum((F.col("2018") > 40).cast("int")).alias("above_40_in_2018"),
                F.sum((F.col("2019") > 40).cast("int")).alias("above_40_in_2019")).show()

<a id='3.3'></a>
## 3.3 Madrid Central: Pza. del Carmen
Madrid Central, a low-emission zone in the city centre, started on 30 November 2018. Station 35 (Pza. del Carmen) is inside it. A drop is *consistent with* the measure, not proof of it: weather changes NO₂ a lot from one year to another.

In [ ]:
(no2DF.where("STATION = 35")
   .groupBy("YEAR").pivot("MONTH").agg(F.round(F.avg("VALUE"), 1))
   .orderBy("YEAR").toPandas())

<a id='3.4'></a>
## 3.4 The hourly limit
An hourly NO₂ value above **200 µg/m³** may happen at most **18 times per calendar year** at each station.

In [ ]:
(no2DF.where("VALUE > 200")
   .groupBy("STATION").pivot("YEAR", [2018, 2019]).count()
   .join(F.broadcast(stationsDF), "STATION")
   .orderBy(F.desc("2019"))
   .toPandas())

<a id='3.5'></a>
## 3.5 Hours, weekdays and seasons
`H01` is 00:00–01:00 and `H24` is 23:00–24:00, Madrid local time.

In [ ]:
no2DF.groupBy("HOUR").agg(F.round(F.avg("VALUE"), 1).alias("NO2")).orderBy("HOUR").toPandas().T

In [ ]:
(no2DF
   .withColumn("WEEKDAY", F.date_format(F.make_date("YEAR", "MONTH", "DAY"), "E"))
   .groupBy("WEEKDAY").agg(F.round(F.avg("VALUE"), 1).alias("NO2"))
   .orderBy("NO2").show())

In [ ]:
# NO2 (8) peaks in winter, ozone (14) in summer
(validDF.where("MAGNITUDE IN (8, 14)")
   .groupBy("MONTH").pivot("MAGNITUDE", [8, 14]).agg(F.round(F.avg("VALUE"), 1))
   .orderBy("MONTH").toPandas())

<a id='3.6'></a>
## 3.6 Save the result as Parquet
From the **raw** zone (CSV) to a **curated** zone: long, validated, typed, columnar and partitioned by year.

In [ ]:
CURATED = "hdfs://localhost:9000/datalake/curated/madrid/airquality_hourly"
validDF.write.mode("overwrite").partitionBy("YEAR").parquet(CURATED)

aq19 = spark.read.parquet(CURATED).where("YEAR = 2019")
aq19.explain()     # look for PartitionFilters: only YEAR=2019/ is read

<a id='4'></a>
# 4. TearDown

Don't forget to stop the service we've used after lab completion.

<a id='4.1'></a>
## 4.1 Stop Spark and Hadoop
Stop the SparkSession here, then switch Hadoop off in the OSBDET WebUI (`localhost:2026`). Finally, close this notebook with **File → Close and Halt**.

In [ ]:
spark.stop()